<a href="https://colab.research.google.com/github/umachandola249/skills-getting-started-with-github-copilot/blob/main/zomatofile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import zipfile

# Unzip the file
with zipfile.ZipFile('/content/zomato.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')


In [15]:
import pandas as pd
import numpy as np
df = pd.read_csv('/content/zomato.csv')# loading data set

# Clean all DataFrame column names by stripping whitespace
df.columns = df.columns.str.strip()

# select only columns we need for analysis, using cleaned names
column_to_keep = ['name', 'online_order', 'book_table', 'rate', 'votes', 'location',
                  'rest_type', 'cuisines', 'approx_cost(for two people)',
                  'listed_in(type)', 'listed_in(city)']
df = df[column_to_keep]

# clean the 'rate' column (column name is now 'rate' after stripping)
def clean_rate ( value ):
  if pd.isna(value ) or value == 'NEW' or value =='-':
    return  np.nan
  return float ( str (value).split('/')[0].strip ())
df['rate']= df['rate'].apply(clean_rate)

# clean approx_cost column (column name is now 'approx_cost(for two people)')
# Remove commas and convert to numeric
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '').str.strip()
df['approx_cost(for two people)'] = pd.to_numeric(df['approx_cost(for two people)'], errors='coerce')

In [10]:
# Fix the 'approx_cost' column name inconsistency
df.rename(columns={' approx_cost( for two people )': 'approx_cost(for two people)'}, inplace=True)


There is a typo in the column name `locatin`. I will correct it to `location`.

In [11]:
# Fix the typo in the column name 'locatin'
df.rename(columns={' location ': 'location'}, inplace=True)

df = df.dropna( subset =[ 'rate', 'approx_cost(for two people)', 'location'])


DROPE ROWS WITH MISSING VALUES IN CRITICAL COLUMN


In [16]:
df = df.dropna(subset=['rate', 'approx_cost(for two people)', 'location'])

SAVING THE  CLEAN DATA TO NEW CSV FILE

In [17]:
df.to_csv(' zomato_cleaned.csv', index = False)
print ( " Data cleaning complete ! 'zomato_ceaned.csv' is ready." )

 Data cleaning complete ! 'zomato_ceaned.csv' is ready.


# New Section

ANALYTIACL QUERIES WITH SQL  ---

In [20]:
import sqlite3
# connect to temporary database and upload clean data csv
connect = sqlite3.connect('zomato.db') # Corrected database file name
cursor = connect.cursor()
df.to_sql('restaurants', connect, if_exists='replace', index=False) # Corrected table name

41418

QUERY 1 : FINDING TOP FIVE LOCCATION WITH HIGHEST AVERAGE RESTAURANT RATINGS

In [24]:
import sqlite3
import pandas as pd

# Re-establish connection to ensure the latest database state is accessed
connect = sqlite3.connect('zomato.db')

query1= ''' SELECT location , COUNT(*) as total_restaurants, AVG( rate) as
avg_rating
FROM restaurants
GROUP BY location
ORDER BY avg_rating DESC
LIMIT  5 ; '''
print ( '--- TOP LOACTION WITH HIGHEST AVERAGE RATING--- ')
print(pd.read_sql_query(query1, connect))


# comparing average cost based on online ordering capability
query2=''' SELECT online_order , AVG('approx_cost( for two people )')as avg_cost, COUNT(*)as count
FROM restaurants
GROUP  BY online_order;
'''
print( ' \n ---- ONLINE ORDER COST COMPARISION----')
print ( pd.read_sql_query(query2 , connect ))
connect.close()

--- TOP LOACTION WITH HIGHEST AVERAGE RATING--- 
                location  total_restaurants  avg_rating
0           Lavelle Road                481    4.141788
1  Koramangala 3rd Block                191    4.020419
2         St. Marks Road                343    4.017201
3  Koramangala 5th Block               2297    4.006661
4          Church Street                546    3.992125
 
 ---- ONLINE ORDER COST COMPARISION----
  online_order  avg_cost  count
0           No       0.0  14212
1          Yes       0.0  27206
